# Notebook Setup & Imports

In [1]:
import sys
from pathlib import Path

# Add project root to PYTHONPATH
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import torch
import pandas as pd
import numpy as np

# Import Project Modules

In [2]:
from src.dataset import load_data, make_data_loader, make_windows
from src.model import StockMLP, StockCNN
from src.train import train
from src.evaluate import evaluate

# Configuration

In [3]:
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"
BATCH_SIZE = 512
WINDOW_SIZE = 30
EPOCHS = 10
USE_CNN = False  # switch between MLP and CNN
NORMALIZE = True

# Load Dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Please place train.csv inside data/ directory."
    )

df = load_data(DATA_PATH, frac=0.1, random_state=42)
print(f"Rows loaded: {len(df):,}")
# load_data already samples the rows
df.describe()


Rows loaded: 16,826,817


,Date,Open,High,Low,Close,Volume
count,16826817,1.682682e+07,1.682682e+07,1.682682e+07,1.682682e+07,1.682682e+07
mean,2015-03-17 02:57:35.951580,2.687375e+01,2.727164e+01,2.646580e+01,2.686960e+01,1.325638e+06
min,2000-09-18 00:00:00,0.000000e+00,7.900000e-02,0.000000e+00,7.851000e-02,0.000000e+00
25%,2009-11-04 00:00:00,7.049737e+00,7.180000e+00,6.922834e+00,7.050000e+00,3.540000e+04
50%,2016-05-06 00:00:00,1.501628e+01,1.526270e+01,1.477710e+01,1.501599e+01,1.843000e+05
75%,2021-03-24 00:00:00,3.069828e+01,3.117000e+01,3.020000e+01,3.069000e+01,8.167000e+05
max,2024-09-23 00:00:00,8.319200e+02,1.010080e+03,4.320597e+02,4.356200e+02,3.565254e+09
std,NaN,3.618627e+01,3.667960e+01,3.567599e+01,3.617482e+01,7.151557e+06


# Creating Sliding Windows

In [5]:
X, y = make_windows(df, window=WINDOW_SIZE)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive ratio:", y.mean())


X shape: (16799256, 30)
y shape: (16799256,)
Positive ratio: 0.500000773843794


# Train / Validation Split

In [ ]:
split_idx = int(len(X) * 0.8)

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Total samples: {len(X):,}")
print(f"Train samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")


Total samples: 16,799,256
Train samples: 839,962
Validation samples: 15,959,294


# DataLoaders

In [7]:
from torch.utils.data import TensorDataset, DataLoader

if NORMALIZE:
    from src.features import normalize
    X_train = normalize(X_train)
    X_val = normalize(X_val)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Initialize Model

In [8]:
if USE_CNN:
    model = StockCNN()
    print("Using CNN model")
else:
    model = StockMLP(input_dim=WINDOW_SIZE)
    print("Using MLP model")

model

Using MLP model


StockMLP(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

# Train Model

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()
train(
    model=model,
    loader=train_loader,
    epochs=EPOCHS,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
)


Using device: cuda
Epoch 1/10 - Loss: 0.5441
Epoch 2/10 - Loss: 0.5305
Epoch 3/10 - Loss: 0.5268
Epoch 4/10 - Loss: 0.5247
Epoch 5/10 - Loss: 0.5235
Epoch 6/10 - Loss: 0.5224
Epoch 7/10 - Loss: 0.5219
Epoch 8/10 - Loss: 0.5214
Epoch 9/10 - Loss: 0.5208
Epoch 10/10 - Loss: 0.5202


# Evaluate on Validation Set

In [10]:
model.to(device)

val_accuracy = evaluate(
    model,
    X_val,
    y_val,
    device=device,
    batch_size=512,
)

print(f"Validation Accuracy: {val_accuracy:.4f}")


Validation Accuracy: 0.7429


# Save Trained Model

In [11]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "stock_model_v1.pt"
torch.save(model.state_dict(), model_path)

print(f"Model saved to: {model_path}")

Model saved to: /home/mega/projects/forth-year/NN/stock-trend/models/stock_model_v1.pt


# Quick Sanity Prediction

In [12]:
model.eval()

sample = torch.tensor(X_val[:5], dtype=torch.float32).to(device)
with torch.no_grad():
    logits = model(sample)
    probs = torch.sigmoid(logits)

print("Predicted probabilities:", probs.cpu().numpy())
print("True labels:", y_val[:5])

Predicted probabilities: [0.7808239  0.23408306 0.85244215 0.4242904  0.74686044]
True labels: [1 0 0 0 1]
